In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix

# 1. Load data
df = pd.read_csv('Telco-Customer-Churn.csv')

# 2. Clean TotalCharges (convert empty → NaN → fill with 0 or median)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)   # or df['TotalCharges'].median()

# 3. Encode target
df['churn'] = (df['Churn'] == 'Yes').astype(int)   # cleaner way

# 4. Define features and target
X = df.drop(columns=['customerID', 'Churn', 'churn'])  # drop ID & original target
y = df['churn']

# 5. Identify categorical vs numeric columns
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()

# 6. Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
    ])

# 7. Train-test split (do BEFORE preprocessing to avoid leakage)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 8. Logistic Regression pipeline
log_pipe = Pipeline([
    ('prep', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])
log_pipe.fit(X_train, y_train)
log_pred = log_pipe.predict(X_test)

# 9. KNN pipeline
knn_pipe = Pipeline([
    ('prep', preprocessor),
    ('model', KNeighborsClassifier(n_neighbors=5))
])
knn_pipe.fit(X_train, y_train)
knn_pred = knn_pipe.predict(X_test)

# 10. Results
print("Logistic Regression Results:")
print(classification_report(y_test, log_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, log_pred))

print("\nKNN (k=5) Results:")
print(classification_report(y_test, knn_pred))

Logistic Regression Results:
              precision    recall  f1-score   support

           0       0.85      0.90      0.87      1035
           1       0.66      0.56      0.61       374

    accuracy                           0.81      1409
   macro avg       0.76      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409

Confusion Matrix:
[[927 108]
 [164 210]]

KNN (k=5) Results:
              precision    recall  f1-score   support

           0       0.84      0.84      0.84      1035
           1       0.56      0.56      0.56       374

    accuracy                           0.77      1409
   macro avg       0.70      0.70      0.70      1409
weighted avg       0.77      0.77      0.77      1409

